# Measure Faithfulness Recipe

Faithfulness metrics ask whether an intervention changes the output in the way the explanation claims it should. This notebook sketches three compact metrics: output drop, ROAD-style ranked removal, and locality. It also calls `tl.validate(scope="intervention")` before running the metrics.


In [ ]:
from __future__ import annotations

from collections.abc import Callable
from typing import Any

import torch
from torch import nn

import torchlens as tl


class EvidenceModel(nn.Module):
    """Toy model where selected hidden features support one target logit."""

    def __init__(self) -> None:
        """Initialize deterministic feature and head weights."""

        super().__init__()
        self.features = nn.Linear(5, 5, bias=False)
        self.head = nn.Linear(5, 3, bias=False)
        with torch.no_grad():
            self.features.weight.copy_(torch.eye(5))
            self.head.weight.zero_()
            self.head.weight[0, 0] = 1.0
            self.head.weight[0, 1] = 0.5
            self.head.weight[1, 3] = 1.0
            self.head.weight[2, 4] = 1.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map evidence features to logits.

        Parameters
        ----------
        x:
            Batch of five-dimensional evidence features.

        Returns
        -------
        torch.Tensor
            Three class logits.
        """

        return self.head(torch.relu(self.features(x)))


model = EvidenceModel().eval()
x = torch.tensor([[2.0, 1.0, 0.0, 0.2, 0.1]])
validation_report = tl.validate(model, x, scope="intervention")
print(validation_report)
assert validation_report.invariance

In [ ]:
clean_log = tl.log_forward_pass(model, x, vis_opt="none", intervention_ready=True)
clean_logits = clean_log.layer_list[-1].activation.detach()
target_class = int(clean_logits.argmax(dim=-1).item())
print("clean logits", clean_logits, "target", target_class)

`output_drop` is the simplest intervention metric: ablate the proposed explanatory site and measure how much the target logit drops.


In [ ]:
ablate_log = clean_log.fork("zero_relu")
ablate_log.attach_hooks(tl.func("relu"), tl.zero_ablate()).replay()
ablate_logits = ablate_log.layer_list[-1].activation.detach()
output_drop = clean_logits[0, target_class] - ablate_logits[0, target_class]
print("output_drop", float(output_drop))
assert output_drop > 0

ROAD removes features in ranked order. In real studies the ranking might come from saliency, attribution, or a learned explanation. Here the ranking is the hidden activation magnitude so the example remains self-contained.


In [ ]:
def keep_unremoved_features(mask: torch.Tensor) -> Callable[..., torch.Tensor]:
    """Create a hook that keeps unremoved hidden features.

    Parameters
    ----------
    mask:
        One-dimensional binary feature mask.

    Returns
    -------
    Callable[..., torch.Tensor]
        TorchLens hook callable.
    """

    def _hook(activation: torch.Tensor, *, hook: Any) -> torch.Tensor:
        """Apply the ROAD mask to the activation."""

        return activation * mask.to(device=activation.device, dtype=activation.dtype)

    return _hook


hidden = clean_log.find_sites(tl.func("relu")).first().activation.detach()[0]
ranking = torch.argsort(hidden.abs(), descending=True)
road_scores: dict[int, float] = {}
for removed_count in (1, 2, 3):
    keep_mask = torch.ones_like(hidden)
    keep_mask[ranking[:removed_count]] = 0.0
    road_log = clean_log.fork(f"road_remove_{removed_count}")
    road_log.attach_hooks(tl.func("relu"), keep_unremoved_features(keep_mask)).replay()
    road_logits = road_log.layer_list[-1].activation.detach()
    road_scores[removed_count] = float(clean_logits[0, target_class] - road_logits[0, target_class])

print(road_scores)
assert road_scores[3] >= road_scores[1]

A locality check asks whether an intervention leaves unrelated outputs comparatively stable. This toy model has separate hidden features for the non-target classes, so ablating target evidence should move the target more than an unrelated class.


In [ ]:
locality_ratio = float(
    (clean_logits[0, target_class] - ablate_logits[0, target_class]).abs()
    / ((clean_logits[0, 2] - ablate_logits[0, 2]).abs() + 1e-6)
)
print("locality_ratio", locality_ratio)
assert locality_ratio > 10.0